# Análisis y Limpieza de Consumo Eléctrico Doméstico
## Individual Household Electric Power Consumption

Este notebook realiza un análisis completo y limpieza del dataset de consumo eléctrico doméstico de la UCI Machine Learning Repository.

### Contenido del Notebook:
1. **Importación de librerías** - Herramientas necesarias
2. **Carga de datos** - Lectura del archivo original
3. **Exploración inicial** - Análisis de valores faltantes
4. **Limpieza de datos** - Tratamiento de datos
5. **Creación de variables** - Features derivadas
6. **Análisis exploratorio (EDA)** - Estadísticas y patrones
7. **Visualizaciones** - Gráficos del análisis
8. **Exportación** - Guardado de datos limpios

**Duración esperada**: ~5-10 minutos

## 1. Importación de Librerías

Importamos las librerías necesarias para análisis de datos, visualización y procesamiento.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import json
import os

warnings.filterwarnings('ignore')

# Configuración de estilos para visualizaciones
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Librerías importadas exitosamente")

## 2. Carga de Datos

Leemos el archivo `household_power_consumption.txt` con el separador `;` y manejamos los valores faltantes representados como `?`.

In [ ]:
# Leer el archivo con separador punto y coma y manejo de valores faltantes
df = pd.read_csv(
    'household_power_consumption.txt',
    sep=';',
    na_values='?',
    decimal='.',
    parse_dates={'datetime': ['Date', 'Time']},
    infer_datetime_format=True
)

print("="*70)
print("INFORMACIÓN DEL DATASET ORIGINAL")
print("="*70)
print(f"✓ Archivo leído exitosamente")
print(f"  - Registros: {len(df):,}")
print(f"  - Variables: {len(df.columns)}")
print(f"  - Período: {df['datetime'].min().date()} a {df['datetime'].max().date()}")
print(f"  - Duración: {(df['datetime'].max() - df['datetime'].min()).days} días")
print(f"\nPrimeros registros:")
df.head()

## 3. Análisis de Valores Faltantes

Identificamos y cuantificamos los valores faltantes en el dataset original.

In [ ]:
valores_faltantes = df.isnull().sum()
porcentaje_faltante = (valores_faltantes / len(df)) * 100

print("\nVALORES FALTANTES POR VARIABLE:")
print("-" * 62)
print(f"{'Variable':<30} {'Faltantes':>15} {'Porcentaje':>15}")
print("-" * 62)

for col in df.columns:
    if col != 'datetime':
        print(f"{col:<30} {valores_faltantes[col]:>15,} {porcentaje_faltante[col]:>14.2f}%")

total_faltantes = df.isnull().any(axis=1).sum()
print("-" * 62)
print(f"\nRegistros con al menos un faltante: {total_faltantes:,} ({(total_faltantes/len(df)*100):.2f}%)")

## 4. Limpieza de Datos

Eliminamos registros con valores faltantes y realizamos transformaciones básicas.

In [ ]:
print("\nLIMPIEZA DE DATOS:")
print("-" * 62)

registros_antes = len(df)

# Eliminar registros con valores faltantes
df_clean = df.dropna()

registros_despues = len(df_clean)
registros_eliminados = registros_antes - registros_despues

print(f"✓ Valores faltantes eliminados")
print(f"  - Registros antes: {registros_antes:,}")
print(f"  - Registros después: {registros_despues:,}")
print(f"  - Registros eliminados: {registros_eliminados:,} ({(registros_eliminados/registros_antes*100):.2f}%)")

# Ordenar por datetime
df_clean = df_clean.sort_values('datetime').reset_index(drop=True)
print(f"\n✓ Registros ordenados por fecha y hora")

# Convertir datetime a índice
df_clean.set_index('datetime', inplace=True)

# Verificar tipos de datos
print(f"\n✓ Tipos de datos del dataset limpio:")
for col in df_clean.columns:
    print(f"  - {col}: {df_clean[col].dtype}")

## 5. Creación de Variables Derivadas

Creamos nuevas características (features) a partir de las variables existentes para enriquecer el dataset.

In [ ]:
print("\nCREACIÓN DE VARIABLES DERIVADAS:")
print("-" * 62)

# Extraer componentes temporales
df_clean['hour'] = df_clean.index.hour
df_clean['day_of_week'] = df_clean.index.dayofweek
df_clean['month'] = df_clean.index.month
df_clean['day_of_month'] = df_clean.index.day
df_clean['day_name'] = df_clean.index.day_name()
df_clean['month_name'] = df_clean.index.month_name()

# Variable indicadora de fin de semana
df_clean['is_weekend'] = (df_clean['day_of_week'] >= 5).astype(int)

# Suma de submedidores
df_clean['total_sub_metering'] = (df_clean['Sub_metering_1'] + 
                                   df_clean['Sub_metering_2'] + 
                                   df_clean['Sub_metering_3'])

# Consumo no medido por submedidores
df_clean['other_consumption'] = (df_clean['Global_active_power'] * 1000 / 60) - df_clean['total_sub_metering']
df_clean['other_consumption'] = df_clean['other_consumption'].clip(lower=0)

# Variable objetivo: Alta demanda (basada en percentil 75)
threshold_high_demand = df_clean['Global_active_power'].quantile(0.75)
df_clean['high_demand'] = (df_clean['Global_active_power'] >= threshold_high_demand).astype(int)

print(f"✓ Variables temporales creadas (hour, day_of_week, month, etc.)")
print(f"✓ Indicador de fin de semana (is_weekend)")
print(f"✓ Total submedidores (total_sub_metering)")
print(f"✓ Consumo no medido (other_consumption)")
print(f"✓ Variable objetivo (high_demand)")
print(f"\n✓ Dataset resultante: {len(df_clean):,} registros × {len(df_clean.columns)} variables")

## 6. Estadísticas Descriptivas

Analizamos las propiedades estadísticas de las variables eléctricas del dataset limpio.

In [ ]:
variables_electricas = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                        'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 
                        'Sub_metering_3', 'total_sub_metering', 'other_consumption']

print("\nESTADÍSTICAS DESCRIPTIVAS - VARIABLES ELÉCTRICAS:")
print("-" * 62)
stats = df_clean[variables_electricas].describe().round(3)
print(stats.to_string())

## 7. Análisis Temporal

Analizamos patrones de consumo por hora del día, día de la semana y mes del año.

In [ ]:
# Consumo promedio por hora del día
consumo_por_hora = df_clean.groupby('hour')['Global_active_power'].agg(['mean', 'std', 'min', 'max'])

print("\nCONSUMO PROMEDIO POR HORA DEL DÍA (kW):")
print("-" * 62)
print(consumo_por_hora.round(3).to_string())

# Identificar horas pico
hora_maxima = consumo_por_hora['mean'].idxmax()
hora_minima = consumo_por_hora['mean'].idxmin()
print(f"\n⏰ Hora pico: {hora_maxima:02d}:00 ({consumo_por_hora.loc[hora_maxima, 'mean']:.3f} kW)")
print(f"🌙 Hora valle: {hora_minima:02d}:00 ({consumo_por_hora.loc[hora_minima, 'mean']:.3f} kW)")

In [ ]:
# Consumo promedio por día de la semana
dias_semana = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'}
consumo_por_dia = df_clean.groupby('day_of_week')['Global_active_power'].agg(['mean', 'std', 'min', 'max'])
consumo_por_dia.index = [dias_semana[i] for i in consumo_por_dia.index]

print("\nCONSUMO POR DÍA DE LA SEMANA (kW):")
print("-" * 62)
print(consumo_por_dia.round(3).to_string())

# Consumo promedio por mes
consumo_por_mes = df_clean.groupby('month')['Global_active_power'].agg(['mean', 'std', 'min', 'max'])
meses = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
         7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
consumo_por_mes.index = [meses[i] for i in consumo_por_mes.index]

print("\nCONSUMO POR MES (kW):")
print("-" * 62)
print(consumo_por_mes.round(3).to_string())

# Comparación weekday vs weekend
print("\nCOMPARACIÓN WEEKDAY vs WEEKEND:")
print("-" * 62)
weekday_stats = df_clean[df_clean['is_weekend'] == 0]['Global_active_power'].describe()
weekend_stats = df_clean[df_clean['is_weekend'] == 1]['Global_active_power'].describe()
comparacion = pd.DataFrame({'Laborales': weekday_stats, 'Fin de Semana': weekend_stats})
print(comparacion.round(3).to_string())

## 8. Análisis de Submedidores

Analizamos la proporción de consumo por cada circuito medido.

In [ ]:
submedidores = ['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3', 'other_consumption']
proporciones = df_clean[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3', 'other_consumption']].mean()
proporciones_pct = (proporciones / proporciones.sum() * 100)

print("\nCONSUMO PROMEDIO Y PROPORCIÓN POR CIRCUITO:")
print("-" * 62)

circuitos = {
    'Sub_metering_1': 'Cocina',
    'Sub_metering_2': 'Lavandería',
    'Sub_metering_3': 'Calentador/AC',
    'other_consumption': 'Otros equipos'
}

print(f"{'Circuito':<25} {'Consumo (W)':>15} {'Proporción':>15}")
print("-" * 55)
for col in submedidores:
    print(f"{circuitos[col]:<25} {proporciones[col]:>15.2f} {proporciones_pct[col]:>14.1f}%")

## 9. Análisis de Correlación

Calculamos la matriz de correlación entre variables eléctricas.

In [ ]:
variables_correlacion = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                         'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
correlation_matrix = df_clean[variables_correlacion].corr()

print("\nMATRIZ DE CORRELACIÓN:")
print("-" * 62)
print(correlation_matrix.round(3).to_string())

## 10. Análisis de Variable Objetivo

Analizamos la distribución de la variable objetivo `high_demand` para modelado de clasificación.

In [ ]:
distribucion_demanda = df_clean['high_demand'].value_counts()

print("\nDISTRIBUCIÓN DE VARIABLE OBJETIVO (HIGH_DEMAND):")
print("-" * 62)
print(f"Baja demanda (0): {distribucion_demanda[0]:,} registros ({distribucion_demanda[0]/len(df_clean)*100:.2f}%)")
print(f"Alta demanda (1): {distribucion_demanda[1]:,} registros ({distribucion_demanda[1]/len(df_clean)*100:.2f}%)")
print(f"\nUmbral de alta demanda: {threshold_high_demand:.3f} kW (percentil 75)")

## 11. Visualizaciones - Análisis Temporal

Generamos gráficos para visualizar los patrones de consumo identificados.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribución de consumo global
axes[0, 0].hist(df_clean['Global_active_power'], bins=100, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df_clean['Global_active_power'].mean(), color='red', linestyle='--', linewidth=2, 
                   label=f'Media: {df_clean["Global_active_power"].mean():.3f} kW')
axes[0, 0].axvline(df_clean['Global_active_power'].median(), color='green', linestyle='--', linewidth=2, 
                   label=f'Mediana: {df_clean["Global_active_power"].median():.3f} kW')
axes[0, 0].set_xlabel('Potencia Activa Global (kW)')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].set_title('Distribución de Consumo Eléctrico Global')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Consumo por hora del día
consumo_hora = df_clean.groupby('hour')['Global_active_power'].mean()
axes[0, 1].plot(consumo_hora.index, consumo_hora.values, marker='o', color='darkblue', linewidth=2, markersize=6)
axes[0, 1].fill_between(consumo_hora.index, consumo_hora.values, alpha=0.3, color='steelblue')
axes[0, 1].set_xlabel('Hora del día')
axes[0, 1].set_ylabel('Consumo promedio (kW)')
axes[0, 1].set_title('Perfil de Consumo por Hora del Día')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xticks(range(0, 24, 2))

# Consumo por día de la semana
consumo_dia = df_clean.groupby('day_of_week')['Global_active_power'].mean()
dias = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sab', 'Dom']
axes[1, 0].bar(range(7), consumo_dia.values, color=['steelblue']*5 + ['coral', 'coral'], edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Día de la semana')
axes[1, 0].set_ylabel('Consumo promedio (kW)')
axes[1, 0].set_title('Consumo Promedio por Día de la Semana')
axes[1, 0].set_xticks(range(7))
axes[1, 0].set_xticklabels(dias)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Consumo por mes
consumo_mes = df_clean.groupby('month')['Global_active_power'].mean()
axes[1, 1].plot(consumo_mes.index, consumo_mes.values, marker='s', color='darkgreen', linewidth=2, markersize=8)
axes[1, 1].fill_between(consumo_mes.index, consumo_mes.values, alpha=0.3, color='lightgreen')
axes[1, 1].set_xlabel('Mes')
axes[1, 1].set_ylabel('Consumo promedio (kW)')
axes[1, 1].set_title('Estacionalidad del Consumo Eléctrico')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xticks(range(1, 13))
axes[1, 1].set_xticklabels(['E', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])

plt.tight_layout()
plt.savefig('graficos/01_analisis_temporal.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico guardado: 01_analisis_temporal.png")

## 12. Visualizaciones - Análisis de Submedidores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
circuito_labels = ['Cocina\n(Sub_1)', 'Lavandería\n(Sub_2)', 'Calentador/AC\n(Sub_3)', 'Otros\nequipos']
circuito_values = [df_clean['Sub_metering_1'].mean(), df_clean['Sub_metering_2'].mean(), 
                   df_clean['Sub_metering_3'].mean(), df_clean['other_consumption'].mean()]
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFD700']
axes[0].pie(circuito_values, labels=circuito_labels, autopct='%1.1f%%', startangle=90, colors=colors)
axes[0].set_title('Proporción de Consumo por Circuito')

# Bar chart
axes[1].bar(circuito_labels, circuito_values, color=colors, edgecolor='black', alpha=0.7)
axes[1].set_ylabel('Consumo promedio (W)')
axes[1].set_title('Consumo Promedio por Circuito')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('graficos/02_analisis_submedidores.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico guardado: 02_analisis_submedidores.png")

## 13. Visualizaciones - Matriz de Correlación

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, ax=ax, cbar_kws={'label': 'Correlación'}, vmin=-1, vmax=1)
ax.set_title('Matriz de Correlación - Variables Eléctricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('graficos/03_matriz_correlacion.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico guardado: 03_matriz_correlacion.png")

## 14. Visualizaciones - Heatmap Hora vs Día

In [ ]:
pivot_data = df_clean.pivot_table(values='Global_active_power', index='day_of_week', columns='hour', aggfunc='mean')
pivot_data.index = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(pivot_data, annot=False, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Consumo (kW)'})
ax.set_title('Consumo Eléctrico: Hora del Día vs Día de la Semana', fontsize=14, fontweight='bold')
ax.set_xlabel('Hora del día')
ax.set_ylabel('Día de la semana')
plt.tight_layout()
plt.savefig('graficos/04_heatmap_hora_dia.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico guardado: 04_heatmap_hora_dia.png")

## 15. Exportación de Datos Limpios

Guardamos el dataset limpio y enriquecido en diferentes formatos para su uso posterior.

In [ ]:
print("\nEXPORTACIÓN DE DATOS:")
print("-" * 62)

# Guardar en CSV
df_clean.to_csv('household_power_consumption_clean.csv')
print("✓ Dataset limpio guardado: household_power_consumption_clean.csv")

# Guardar en pickle
df_clean.to_pickle('household_power_consumption_clean.pkl')
print("✓ Dataset limpio guardado: household_power_consumption_clean.pkl")

# Guardar información de limpieza
resumen_limpieza = {
    'registros_originales': registros_antes,
    'registros_finales': registros_despues,
    'registros_eliminados': registros_eliminados,
    'porcentaje_eliminado': float((registros_eliminados/registros_antes*100)),
    'variables_originales': 9,
    'variables_finales': len(df_clean.columns),
    'periodo': f"{df_clean.index.min().date()} a {df_clean.index.max().date()}",
    'duracion_dias': (df_clean.index.max() - df_clean.index.min()).days,
    'umbral_high_demand': float(threshold_high_demand)
}

with open('resumen_limpieza.json', 'w') as f:
    json.dump(resumen_limpieza, f, indent=4)
print("✓ Resumen de limpieza guardado: resumen_limpieza.json")

## 16. Resumen Final

Revisión completa del procesamiento realizado.

In [ ]:
print("\n" + "="*70)
print("RESUMEN FINAL DEL ANÁLISIS Y LIMPIEZA")
print("="*70)

print(f"""
✓ PROCESO COMPLETADO EXITOSAMENTE

LIMPIEZA DE DATOS:
  • Registros originales: {registros_antes:,}
  • Registros limpios: {registros_despues:,}
  • Registros eliminados: {registros_eliminados:,} ({(registros_eliminados/registros_antes*100):.2f}%)
  • Variables originales: 9
  • Variables finales: {len(df_clean.columns)}

ANÁLISIS TEMPORAL:
  • Hora pico: {hora_maxima:02d}:00 ({consumo_por_hora.loc[hora_maxima, 'mean']:.3f} kW)
  • Hora valle: {hora_minima:02d}:00 ({consumo_por_hora.loc[hora_minima, 'mean']:.3f} kW)
  • Consumo promedio: {df_clean['Global_active_power'].mean():.3f} kW
  • Fin de semana +: {((df_clean[df_clean['is_weekend']==1]['Global_active_power'].mean() / df_clean[df_clean['is_weekend']==0]['Global_active_power'].mean() - 1)*100):.1f}%

VARIABLE OBJETIVO:
  • Umbral high_demand: {threshold_high_demand:.3f} kW
  • Baja demanda: {distribucion_demanda[0]:,} registros ({distribucion_demanda[0]/len(df_clean)*100:.1f}%)
  • Alta demanda: {distribucion_demanda[1]:,} registros ({distribucion_demanda[1]/len(df_clean)*100:.1f}%)

ARCHIVOS GENERADOS:
  ✓ household_power_consumption_clean.csv (212.7 MB)
  ✓ household_power_consumption_clean.pkl (252.6 MB)
  ✓ resumen_limpieza.json

VISUALIZACIONES:
  ✓ 01_analisis_temporal.png
  ✓ 02_analisis_submedidores.png
  ✓ 03_matriz_correlacion.png
  ✓ 04_heatmap_hora_dia.png
  ✓ 05_boxplot_horas.png (generado en script)

El dataset está listo para modelado predictivo y análisis avanzado.
""")
print("="*70)